In [1]:
# rf_hm0_2005_full_optionA.py
# Pure-numeric, comma-separated text inputs:
#   X_2005.txt: Lat,Lon,u10,v10,MSLP   (5 columns)
#   Y_2005.txt: Hm0                   (1 column, same number of rows)
#
# Trains RandomForestRegressor on a large random sample (reservoir sampling),
# evaluates on a holdout sample, and produces:
#  - metrics.csv
#  - feature_importance.csv + feature_importance.png
#  - obs_vs_pred_ci.png (95% RF interval ribbon + perfect agreement line y=x)
#  - actual_vs_predicted_test_sample.txt (Option A)
#  - Maps: observed/predicted/error_mean/rmse (png)
#
# Run:
#   pip install numpy pandas scikit-learn matplotlib
#   python rf_hm0_2005_full_optionA.py

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# -----------------------------
# USER SETTINGS
# -----------------------------
X_PATH = Path("../../Copy of Dataset and code - Wave Power/X_2005.txt")   # comma-separated, no header
Y_PATH = Path("../../Copy of Dataset and code - Wave Power/Y_2005.txt")   # 1 col, comma-separated (or one-per-line; comma sep still OK)

OUTDIR = Path("rf_2005_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

RANDOM_SEED = 42
CHUNK_ROWS = 300_000          # adjust for RAM (100k–1M typical)

# Sampling sizes (tune)
TRAIN_N = 600_000
TEST_N  = 150_000

# RF hyperparameters (tune)
RF_PARAMS = dict(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    min_samples_leaf=5,
    max_features="sqrt",
)

# Grid resolution (set explicitly if known; else inferred from sample)
DLAT = None   # e.g., 0.25
DLON = None   # e.g., 0.25

# Plot settings
PLOT_SUBSAMPLE = 60_000
CI_BINS = 80
ADD_PM10_ENVELOPE = True

# Option A output (test sample only)
PAIRS_OUT = OUTDIR / "actual_vs_predicted_test_sample.txt"
PAIRS_SEP = "\t"  # tab-separated

In [3]:
# -----------------------------
# STREAM READERS (comma-separated)
# -----------------------------
def stream_chunks(x_path: Path, y_path: Path):
    """Yield aligned chunks (X_df, y_series)."""
    x_iter = pd.read_csv(
        x_path, header=None, chunksize=CHUNK_ROWS,
        sep=",", engine="python"
    )
    y_iter = pd.read_csv(
        y_path, header=None, chunksize=CHUNK_ROWS,
        sep=",", engine="python"
    )

    for Xc, Yc in zip(x_iter, y_iter):
        if Xc.shape[1] < 5:
            raise ValueError(f"X chunk has {Xc.shape[1]} columns; expected 5 (comma-separated).")

        Xc = Xc.iloc[:, :5]
        Xc.columns = ["Lat", "Lon", "u10", "v10", "MSLP"]
        y = Yc.iloc[:, 0].rename("Hm0")

        # numeric + drop NaNs/inf
        Xc = Xc.apply(pd.to_numeric, errors="coerce")
        y = pd.to_numeric(y, errors="coerce")

        m = np.isfinite(Xc.to_numpy()).all(axis=1) & np.isfinite(y.to_numpy())
        yield Xc.loc[m].reset_index(drop=True), y.loc[m].reset_index(drop=True)


In [4]:
# -----------------------------
# FAST RESERVOIR SAMPLING (chunk-wise)
# -----------------------------
def _apply_replacements(res_X, res_y, X_new, y_new, seen_start, rng):
    """
    For a block of new items X_new (m,5), y_new (m,),
    perform reservoir replacements for a reservoir of size k=len(res_y).
    Vectorized, but preserves sequential 'last write wins' behavior for duplicates.
    """
    k = len(res_y)
    m = len(y_new)
    if m == 0:
        return

    total_seen = seen_start + np.arange(1, m + 1, dtype=np.int64)  # seen after each new item
    u = rng.random(m)
    j = (u * total_seen).astype(np.int64)  # uniform int in [0, total_seen-1]

    sel = j < k
    if not np.any(sel):
        return

    j_sel = j[sel]
    i_sel = np.nonzero(sel)[0]

    # keep LAST i for each j (matching sequential replacement)
    order = np.lexsort((i_sel, j_sel))  # sort by j then i
    j_s = j_sel[order]
    i_s = i_sel[order]
    keep_last = np.r_[j_s[1:] != j_s[:-1], True]
    j_u = j_s[keep_last]
    i_u = i_s[keep_last]

    res_X[j_u] = X_new[i_u]
    res_y[j_u] = y_new[i_u]

In [5]:
def reservoir_sample(x_path: Path, y_path: Path, n_total: int, seed: int):
    """One-pass uniform reservoir sample of n_total rows."""
    rng = np.random.default_rng(seed)

    res_X = np.empty((n_total, 5), dtype=np.float32)
    res_y = np.empty((n_total,), dtype=np.float32)

    filled = 0
    seen = 0

    for Xc, yc in stream_chunks(x_path, y_path):
        Xnp = Xc.to_numpy(dtype=np.float32, copy=False)
        ynp = yc.to_numpy(dtype=np.float32, copy=False)
        m = len(ynp)
        if m == 0:
            continue

        if filled < n_total:
            take = min(n_total - filled, m)
            res_X[filled:filled + take] = Xnp[:take]
            res_y[filled:filled + take] = ynp[:take]
            filled += take
            seen += take

            rem = m - take
            if rem > 0:
                _apply_replacements(res_X, res_y, Xnp[take:], ynp[take:], seen_start=seen, rng=rng)
                seen += rem
        else:
            _apply_replacements(res_X, res_y, Xnp, ynp, seen_start=seen, rng=rng)
            seen += m

    if filled < n_total:
        res_X = res_X[:filled]
        res_y = res_y[:filled]

    return res_X, res_y

In [6]:
# -----------------------------
#  OBS vs PRED PLOT (RF interval + 1:1 line)
# -----------------------------
def plot_obs_pred_with_rf_ci(
    rf: RandomForestRegressor,
    X_test: np.ndarray,
    y_test: np.ndarray,
    out_png: Path,
    n_plot: int = 60_000,
    n_bins: int = 80,
    seed: int = 42,
    add_pm10: bool = True,
):
    rng = np.random.default_rng(seed)
    y = np.asarray(y_test, dtype=np.float64)
    n = len(y)

    idx = rng.choice(n, size=min(n_plot, n), replace=False)
    Xp = np.asarray(X_test[idx], dtype=np.float32)
    yt = y[idx]

    # per-tree prediction spread
    tree_preds = np.vstack([est.predict(Xp) for est in rf.estimators_]).astype(np.float64)
    pred_med = np.median(tree_preds, axis=0)
    pred_lo  = np.percentile(tree_preds, 2.5, axis=0)
    pred_hi  = np.percentile(tree_preds, 97.5, axis=0)

    # ribbon by observed bins
    order = np.argsort(yt)
    yt_s  = yt[order]
    med_s = pred_med[order]
    lo_s  = pred_lo[order]
    hi_s  = pred_hi[order]

    edges = np.linspace(0, len(yt_s), n_bins + 1).astype(int)
    x_bin, med_b, lo_b, hi_b = (np.empty(n_bins) for _ in range(4))

    for b in range(n_bins):
        a, c = edges[b], edges[b + 1]
        if c <= a:
            c = a + 1
        sl = slice(a, c)
        x_bin[b] = np.nanmean(yt_s[sl])
        med_b[b] = np.nanmean(med_s[sl])
        lo_b[b]  = np.nanmean(lo_s[sl])
        hi_b[b]  = np.nanmean(hi_s[sl])

    # metrics on plotted subset
    err = yt - pred_med
    mae = np.nanmean(np.abs(err))
    rmse = np.sqrt(np.nanmean(err**2))
    r2 = 1 - (np.nansum(err**2) / np.nansum((yt - np.nanmean(yt))**2))

    fig, ax = plt.subplots(figsize=(7.4, 6.4))

    hb = ax.hexbin(yt, pred_med, gridsize=70, mincnt=1)
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("Count")

    ax.fill_between(x_bin, lo_b, hi_b, alpha=0.25, label="RF 95% interval (tree spread)")
    ax.plot(x_bin, med_b, linewidth=2.0, label="Binned median prediction")

    mn = float(np.nanmin([yt.min(), pred_med.min()]))
    mx = float(np.nanmax([yt.max(), pred_med.max()]))
    ax.plot([mn, mx], [mn, mx], linestyle="--", linewidth=2.0, label="Perfect agreement (y=x)")

    if add_pm10:
        ax.plot([mn, mx], [0.9 * mn, 0.9 * mx], linestyle=":", linewidth=1.5, label="−10%")
        ax.plot([mn, mx], [1.1 * mn, 1.1 * mx], linestyle=":", linewidth=1.5, label="+10%")

    ax.set_xlabel("Observed Hm0")
    ax.set_ylabel("Predicted Hm0")
    ax.set_title("Observed vs Predicted Hm0 (RF, 2005)")
    ax.text(0.02, 0.98, f"MAE={mae:.3f}\nRMSE={rmse:.3f}\nR²={r2:.3f}\n(n={len(yt):,})",
            transform=ax.transAxes, va="top")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(out_png, dpi=300)
    plt.close(fig)


In [7]:
# -----------------------------
# GRID / MAP HELPERS (streaming accumulation)
# -----------------------------
def infer_resolution(values: np.ndarray) -> float:
    u = np.unique(np.round(values.astype(np.float64), 6))
    if u.size < 2:
        return 0.25
    diffs = np.diff(np.sort(u))
    diffs = diffs[diffs > 0]
    return float(np.median(diffs)) if diffs.size else 0.25


def init_grid(lat_min, lat_max, lon_min, lon_max, dlat, dlon):
    nlat = int(np.floor((lat_max - lat_min) / dlat + 0.5)) + 1
    nlon = int(np.floor((lon_max - lon_min) / dlon + 0.5)) + 1

    def zeros():
        return np.zeros((nlat, nlon), dtype=np.float64)

    return {
        "count": zeros(),
        "sum_obs": zeros(),
        "sum_pred": zeros(),
        "sum_err": zeros(),
        "sum_err2": zeros(),
        "lat_min": float(lat_min),
        "lon_min": float(lon_min),
        "dlat": float(dlat),
        "dlon": float(dlon),
        "nlat": nlat,
        "nlon": nlon,
    }


def update_grid(grid, lat, lon, obs, pred):
    lat_min, lon_min = grid["lat_min"], grid["lon_min"]
    dlat, dlon = grid["dlat"], grid["dlon"]
    nlat, nlon = grid["nlat"], grid["nlon"]

    ilat = np.round((lat - lat_min) / dlat).astype(np.int64)
    ilon = np.round((lon - lon_min) / dlon).astype(np.int64)

    ok = (
        (ilat >= 0) & (ilat < nlat) &
        (ilon >= 0) & (ilon < nlon) &
        np.isfinite(obs) & np.isfinite(pred)
    )
    ilat = ilat[ok]
    ilon = ilon[ok]
    obs = obs[ok]
    pred = pred[ok]
    err = obs - pred

    np.add.at(grid["count"], (ilat, ilon), 1.0)
    np.add.at(grid["sum_obs"], (ilat, ilon), obs)
    np.add.at(grid["sum_pred"], (ilat, ilon), pred)
    np.add.at(grid["sum_err"], (ilat, ilon), err)
    np.add.at(grid["sum_err2"], (ilat, ilon), err * err)


def finalize_maps(grid):
    count = grid["count"]
    with np.errstate(invalid="ignore", divide="ignore"):
        obs_mean  = grid["sum_obs"] / count
        pred_mean = grid["sum_pred"] / count
        err_mean  = grid["sum_err"] / count
        rmse      = np.sqrt(grid["sum_err2"] / count)

    obs_mean[count == 0] = np.nan
    pred_mean[count == 0] = np.nan
    err_mean[count == 0] = np.nan
    rmse[count == 0] = np.nan

    lon_min = grid["lon_min"]
    lat_min = grid["lat_min"]
    nlon = grid["nlon"]
    nlat = grid["nlat"]
    dlon = grid["dlon"]
    dlat = grid["dlat"]
    extent = (lon_min, lon_min + (nlon - 1) * dlon, lat_min, lat_min + (nlat - 1) * dlat)

    return obs_mean.astype(np.float32), pred_mean.astype(np.float32), err_mean.astype(np.float32), rmse.astype(np.float32), extent


def save_map(arr, extent, title, path):
    plt.figure()
    plt.imshow(arr, origin="lower", extent=extent, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()

In [8]:
# -----------------------------
# MAIN
# -----------------------------
def main():
    # 1) Reservoir sample train+test
    print("Sampling train+test (one pass reservoir sampling)...")
    Xs, ys = reservoir_sample(X_PATH, Y_PATH, TRAIN_N + TEST_N, seed=RANDOM_SEED)

    # Shuffle and split
    rng = np.random.default_rng(RANDOM_SEED)
    perm = rng.permutation(len(ys))
    Xs = Xs[perm]
    ys = ys[perm]

    Xtr, ytr = Xs[:TRAIN_N], ys[:TRAIN_N]
    Xte, yte = Xs[TRAIN_N:], ys[TRAIN_N:]

    # 2) Train RF
    print("Training Random Forest...")
    rf = RandomForestRegressor(**RF_PARAMS)
    rf.fit(Xtr, ytr)

    # 3) Evaluate on test sample
    pred_te = rf.predict(Xte).astype(np.float32)

    metrics = pd.DataFrame([{
        "MAE": mean_absolute_error(yte, pred_te),
        "RMSE": mean_squared_error(yte, pred_te) ** 0.5,
        "R2": r2_score(yte, pred_te),
        "n_train": int(len(ytr)),
        "n_test": int(len(yte)),
    }])
    metrics.to_csv(OUTDIR / "metrics.csv", index=False)
    print("\n=== Analytics (test sample) ===")
    print(metrics.to_string(index=False))

    # ---- OPTION A: write actual vs predicted (test sample) as text ----
    pairs = pd.DataFrame({"actual": yte.astype(np.float32), "predicted": pred_te.astype(np.float32)})
    pairs.to_csv(PAIRS_OUT, sep=PAIRS_SEP, index=False)
    print(f"\nWrote: {PAIRS_OUT.resolve()}")

    # Feature importance + plot
    feat_names = ["Lat", "Lon", "u10", "v10", "MSLP"]
    fi = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
    fi.to_csv(OUTDIR / "feature_importance.csv")

    plt.figure(figsize=(6.8, 4.4))
    fi.iloc[::-1].plot(kind="barh")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(OUTDIR / "feature_importance.png", dpi=300)
    plt.close()

    # Attractive obs-pred plot with CI + perfect agreement
    plot_obs_pred_with_rf_ci(
        rf=rf,
        X_test=Xte,
        y_test=yte,
        out_png=OUTDIR / "obs_vs_pred_ci.png",
        n_plot=PLOT_SUBSAMPLE,
        n_bins=CI_BINS,
        seed=RANDOM_SEED,
        add_pm10=ADD_PM10_ENVELOPE
    )

    # 4) Grid definition from sample
    lat_s = Xs[:, 0]
    lon_s = Xs[:, 1]
    dlat = DLAT if DLAT is not None else infer_resolution(lat_s)
    dlon = DLON if DLON is not None else infer_resolution(lon_s)

    lat_min, lat_max = float(np.nanmin(lat_s)), float(np.nanmax(lat_s))
    lon_min, lon_max = float(np.nanmin(lon_s)), float(np.nanmax(lon_s))
    grid = init_grid(lat_min, lat_max, lon_min, lon_max, dlat, dlon)
    print(f"\nGrid: dlat={dlat}, dlon={dlon}")

    # 5) Stream full dataset for spatial maps (second pass)
    print("\nStreaming full dataset for maps (second pass)...")
    total = 0
    for Xc, yc in stream_chunks(X_PATH, Y_PATH):
        Xnp = Xc.to_numpy(dtype=np.float32, copy=False)
        obs = yc.to_numpy(dtype=np.float32, copy=False)
        pred = rf.predict(Xnp).astype(np.float32)

        update_grid(
            grid,
            lat=Xnp[:, 0].astype(np.float64),
            lon=Xnp[:, 1].astype(np.float64),
            obs=obs.astype(np.float64),
            pred=pred.astype(np.float64),
        )

        total += len(obs)
        if total % 2_000_000 < len(obs):
            print(f"Processed {total:,} rows")

    obs_map, pred_map, err_map, rmse_map, extent = finalize_maps(grid)

    save_map(obs_map, extent, "Observed Hm0 (2005)", OUTDIR / "map_observed_hm0.png")
    save_map(pred_map, extent, "Predicted Hm0 (RF, 2005)", OUTDIR / "map_predicted_hm0.png")
    save_map(err_map, extent, "Mean Error (Obs - Pred) Hm0", OUTDIR / "map_error_mean.png")
    save_map(rmse_map, extent, "RMSE per Grid Cell (Hm0)", OUTDIR / "map_rmse.png")

    print(f"\nSaved outputs to: {OUTDIR.resolve()}")
    print("Key outputs:")
    print(" - metrics.csv")
    print(" - feature_importance.csv, feature_importance.png")
    print(" - obs_vs_pred_ci.png")
    print(" - actual_vs_predicted_test_sample.txt")
    print(" - map_observed_hm0.png, map_predicted_hm0.png, map_error_mean.png, map_rmse.png")


In [9]:
if __name__ == "__main__":
    main()

Sampling train+test (one pass reservoir sampling)...


Training Random Forest...



=== Analytics (test sample) ===
     MAE     RMSE       R2  n_train  n_test
0.389644 0.570804 0.867637   600000  150000



Wrote: /home/michele/phd_choise/off shore/src/off_shore/rf_2005_outputs/actual_vs_predicted_test_sample.txt



Grid: dlat=0.25, dlon=0.25

Streaming full dataset for maps (second pass)...


Processed 2,000,678 rows


Processed 4,001,361 rows


Processed 6,002,044 rows


Processed 8,002,727 rows



Saved outputs to: /home/michele/phd_choise/off shore/src/off_shore/rf_2005_outputs
Key outputs:
 - metrics.csv
 - feature_importance.csv, feature_importance.png
 - obs_vs_pred_ci.png
 - actual_vs_predicted_test_sample.txt
 - map_observed_hm0.png, map_predicted_hm0.png, map_error_mean.png, map_rmse.png
